# SGP-TTA — inference demo

Adapts a **DRIVE**-trained U-Net (retinal fundus vessels) to unseen target
images with **SGP-TTA** (Skeleton-Guided Progressive Test-Time Adaptation)
and compares the source-only prediction with the adapted prediction side by
side.

The pipeline mirrors the one used in the paper's evaluation (see the
project's `01-tta.ipynb`): a U-Net with `resnet50` encoder, `albumentations`
preprocessing, and an optional intensity inversion so bright-on-dark
targets (e.g. OCTA) match the source distribution.

- `model/DRIVE_unet.pth` — U-Net (resnet50 encoder) trained on DRIVE retinal vessels
- `sample/` — example target-domain images


## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, str(Path.cwd()))
from sgp_tta import SGPTTA

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 2. Config

In [ ]:
CHECKPOINT_PATH = Path('model/DRIVE_unet.pth')
SAMPLE_DIR      = Path('sample')

MODEL_NAME   = 'unet'
ENCODER_NAME = 'resnet50'
IN_CHANNELS  = 3
NUM_CLASSES  = 1
IMAGE_SIZE   = 512

INVERT = False   # OCTA vessels are bright-on-dark; fundus source is dark-on-bright.


## 3. Dataset

Same preprocessing as `code/process/FundusDataset.py`: resize → scale to
`[0, 1]` → to tensor → (optionally) invert intensities. No masks are needed
for inference.


In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, image_paths, image_size, invert=False):
        self.image_paths = list(image_paths)
        self.image_size  = image_size
        self.invert      = invert
        self.transform = A.Compose([
            A.Resize(height=image_size, width=image_size),
            A.Normalize(mean=0.0, std=1.0, max_pixel_value=255.0),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image_rgb = np.array(Image.open(path).convert('RGB'))
        image_tensor = self.transform(image=image_rgb)['image']
        if self.invert:
            image_tensor = 1.0 - image_tensor
        return {
            'image':     image_tensor,
            'image_rgb': image_rgb,
            'path':      str(path),
        }


image_paths = sorted(SAMPLE_DIR.glob('*.png'))
dataset = InferenceDataset(image_paths, IMAGE_SIZE, invert=INVERT)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
print(f'{len(dataset)} sample images | invert={INVERT}')


## 4. Load the source model

In [ ]:
def fresh_model():
    model = smp.Unet(
        encoder_name=ENCODER_NAME,
        encoder_weights=None,
        in_channels=IN_CHANNELS,
        classes=NUM_CLASSES,
        activation=None,
    ).to(device)
    state = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    model.load_state_dict(state, strict=True)
    return model


print(f'checkpoint: {CHECKPOINT_PATH}')
print(f'arch      : smp.{MODEL_NAME} | encoder={ENCODER_NAME} | in={IN_CHANNELS} | classes={NUM_CLASSES}')


## 5. Source-only inference

Direct forward pass, no adaptation. Gradient tracking is disabled and the
model is kept in eval mode — this mirrors the `'Source'` baseline in
`code/01-tta.ipynb`.


In [ ]:
def run_source(model, loader):
    preds = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            img = batch['image'].to(device)
            logits = model(img)
            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
            preds.append(prob)
    return preds


source_probs = run_source(fresh_model(), loader)
print(f'source predictions: {len(source_probs)}')


## 6. SGP-TTA adaptation

`SGPTTA` swaps every `BatchNorm2d` layer for a `ProgBN` layer in place and
unfreezes only the BN affine parameters. Calling `tta(img)` runs one
adaptation step per image and returns adapted logits. The internal
`sample_num` counter progresses across images so the ProgBN warmup
schedule naturally advances through the stream.

Hyperparameters match `code/01-tta.ipynb` (`temperature=5.0`, `lr=1e-3`,
`num_steps=1`, `entropy_weight=1.0`, `skeleton_weight=1.0`,
`dilate_radius=2`). A fresh source model is passed in so the source-only
baseline above stays untouched.


In [ ]:
def run_sgptta(loader):
    tta = SGPTTA(
        model=fresh_model(),
        temperature=5.0,
        lr=1e-3,
        num_steps=1,
        entropy_weight=1.0,
        skeleton_weight=1.0,
        dilate_radius=2,
    )
    preds = []
    for batch in loader:
        img = batch['image'].to(device)
        logits = tta(img)
        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        preds.append(prob)
    print('sgp-tta stats:', tta.get_stats())
    return preds


sgptta_probs = run_sgptta(loader)
print(f'sgp-tta predictions: {len(sgptta_probs)}')


## 7. Compare

In [ ]:
def show_row(image_rgb, src_prob, adapted_prob, threshold=0.5, title=None):
    src_mask     = (src_prob     > threshold).astype(np.uint8)
    adapted_mask = (adapted_prob > threshold).astype(np.uint8)

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    axes[0].imshow(image_rgb)
    axes[0].set_title(title or 'Input')

    axes[1].imshow(src_prob, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('Source · prob.')

    axes[2].imshow(src_mask, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title('Source · mask')

    axes[3].imshow(adapted_prob, cmap='gray', vmin=0, vmax=1)
    axes[3].set_title('SGP-TTA · prob.')

    axes[4].imshow(adapted_mask, cmap='gray', vmin=0, vmax=1)
    axes[4].set_title('SGP-TTA · mask')

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.show()


for i, batch in enumerate(dataset):
    show_row(
        batch['image_rgb'],
        source_probs[i],
        sgptta_probs[i],
        title=f'Input ({Path(batch["path"]).name})',
    )
